# 06. 市场状态可视化仪表盘

**目标**: 提供市场状态的综合可视化展示

## 1. 导入依赖库

导入必要的 Python 库和项目模块：
- `sys`: 用于添加项目路径到 Python 搜索路径
- `pandas`, `numpy`: 数据处理库
- `datetime`: 日期时间处理
- `ConfigManager`: 配置管理器，用于加载 JQData 配置
- `JQDataClient`: 聚宽数据客户端
- `TrendAnalyzer`: 市场趋势分析器
- `ChartEngine`, `MarketGauge`: 可视化工具

In [3]:
# 添加项目根目录到 Python 路径（必须在导入前执行）
import sys
from pathlib import Path
import os

# 自动检测项目根目录
current_dir = Path.cwd()
project_root = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'core').exists() and (parent / 'config').exists():
        project_root = parent
        break

if project_root is None:
    # 回退到默认路径
    project_root = Path('/home/taotao/dev/QuantTest/TRQuant')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f'✅ 项目根目录已添加到路径: {project_root}')

# 统一环境初始化（自动检测项目路径）
from notebooks.lib import (
    setup_research_environment,
    ErrorBoundary,
    ResultSaver
)

# 初始化研究环境
env = setup_research_environment(verbose=True)

# 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 导入可视化组件
from core.visualization.chart_engine import ChartEngine
from core.visualization.dashboard import MarketGauge

print('✅ 模块加载完成')

2026-01-01 20:09:31,171 - notebooks.lib.research_init - INFO - ✅ 项目根目录: /home/taotao/dev/QuantTest/TRQuant
2026-01-01 20:09:31,176 - notebooks.lib.research_init - INFO - ✅ 加载配置: /home/taotao/dev/QuantTest/TRQuant/notebooks/research/research.yaml


研究环境状态
项目根目录: /home/taotao/dev/QuantTest/TRQuant
Python 版本: 3.12.3
当前时间: 2026-01-01 20:09:31
JQData 客户端: ⏳ 未初始化
趋势分析器: ⏳ 未初始化
评估引擎: ⏳ 未初始化
✅ 模块加载完成


## 2. 初始化数据客户端

初始化配置管理器和 JQData 客户端：
1. 创建 `ConfigManager` 实例
2. 加载 JQData 配置（用户名、密码等）
3. 创建 `JQDataClient` 实例并认证
4. 创建 `TrendAnalyzer` 实例用于市场分析

In [4]:
# 从环境获取组件（带错误处理）
with ErrorBoundary("初始化JQData客户端和TrendAnalyzer") as eb:
    jq = env.get_jqdata_client()
    trend_analyzer = env.get_trend_analyzer()

if eb.has_error:
    print(f"⚠️ 初始化失败: {eb.error_message}")
    jq = None
    trend_analyzer = None
else:
    print('✅ 初始化完成')

2026-01-01 20:09:53,288 - config.config_manager - INFO - 加载配置成功: jqdata_config.json
2026-01-01 20:09:53,789 - jqdata.auth - INFO - 聚宽认证成功: 13327806797
2026-01-01 20:09:53,789 - jqdata.client - INFO - 正在检测账号数据权限...


auth success 


2026-01-01 20:09:54,005 - jqdata.client - INFO - ✅ 检测到实时账号权限: 数据模式: 实时, 范围: 2021-01-02 至 2026-01-01
2026-01-01 20:09:54,005 - notebooks.lib.research_init - INFO - ✅ JQData 客户端初始化成功
2026-01-01 20:09:54,007 - notebooks.lib.research_init - INFO - ✅ TrendAnalyzer 初始化成功


✅ 初始化完成


## 📊 市场状态定义详解

### 一、综合得分 (composite_score)

综合得分范围：**-100 到 +100**

| 得分范围 | 趋势方向 | 颜色 | 含义 |
|---------|---------|------|------|
| > 60 | 强势上涨 | 🟢 深绿 | 趋势强劲，可满仓持有 |
| 30~60 | 上涨趋势 | 🟢 绿色 | 趋势向上，高仓位操作 |
| 10~30 | 弱势上涨 | 🟡 浅绿 | 趋势偏强，谨慎持有 |
| -10~10 | 震荡盘整 | 🟡 黄色 | 无明显趋势，观望为主 |
| -30~-10 | 弱势下跌 | 🟠 橙色 | 趋势偏弱，减仓观望 |
| -60~-30 | 下跌趋势 | 🔴 红色 | 趋势向下，低仓位防守 |
| < -60 | 强势下跌 | 🔴 深红 | 趋势极弱，空仓观望 |

**得分计算方式**: 综合得分 = 短期得分 × 20% + 中期得分 × 30% + 长期得分 × 50%

---

### 二、市场阶段 (market_phase)

市场阶段共14种，分为牛市系列、熊市系列、震荡系列：

#### 🟢 牛市系列（5种）

| 阶段 | 条件 | 建议仓位 | 操作策略 |
|------|------|---------|---------|
| **牛市确认(全周期共振)** | 短+中+长期全部看涨 | 100% | 全仓持有，追强势股 |
| **牛市确认** | 长期>30, 中期>20, 短期>0 | 80% | 高仓位持有 |
| **牛市震荡** | 长期>30, 中期>0 | 60% | 持仓观望，等待方向 |
| **牛市短期调整** | 长期>30, 短期<-20 | 50% | 短期调整，不追高 |
| **牛市中期调整** | 长期>30, 其他情况 | 50% | 逢低布局 |

#### 🔴 熊市系列（5种）

| 阶段 | 条件 | 建议仓位 | 操作策略 |
|------|------|---------|---------|
| **熊市确认(全周期共振)** | 短+中+长期全部看跌 | 0-10% | 空仓观望 |
| **熊市确认** | 长期<-30, 中期<-20, 短期<0 | 10% | 防守为主 |
| **熊市反弹** | 长期<-30, 中期<0 | 20% | 反弹减仓 |
| **熊市技术反弹** | 长期<-30, 短期>20 | 30% | 反弹出货 |
| **熊市筑底** | 长期<-30, 其他情况 | 20% | 等待确认 |

#### 🟡 震荡系列（4种）

| 阶段 | 条件 | 建议仓位 | 操作策略 |
|------|------|---------|---------|
| **突破在即** | 震荡中，全周期转多 | 60% | 准备加仓 |
| **破位风险** | 震荡中，全周期转空 | 30% | 准备减仓 |
| **复苏初期** | 短期>20, 中期>0 | 50% | 逐步建仓 |
| **见顶回落** | 短期<-20, 中期<0 | 30% | 逐步减仓 |
| **窄幅震荡** | |短期|<15, |中期|<15 | 40% | 高抛低吸 |
| **宽幅震荡** | 其他情况 | 40% | 区间操作 |

---

### 三、周期定义

| 周期 | 时间范围 | 权重 | 说明 |
|------|---------|------|------|
| 短期 | 40天 | 20% | 反映近期市场情绪 |
| 中期 | 120天 | 30% | 反映中期趋势 |
| 长期 | 240天 | 50% | 反映长期趋势（最重要） |

---

### 四、仓位建议参考

| 综合得分 | 建议仓位 | 说明 |
|---------|---------|------|
| > 50 | 80-100% | 强势市场，进攻配置 |
| 20~50 | 60-80% | 偏强市场，积极配置 |
| 0~20 | 40-60% | 中性市场，均衡配置 |
| -20~0 | 20-40% | 偏弱市场，防守配置 |
| < -20 | 0-20% | 弱势市场，空仓或极低仓位 |


## 3. 获取当前市场状态

分析当前市场趋势：
- 使用上证指数（000001.XSHG）作为分析标的
- 调用 `TrendAnalyzer.analyze_market()` 获取市场分析结果
- 输出综合得分和市场阶段判断

In [5]:
# 获取当前市场状态（带错误处理）
INDEX_CODE = '000001.XSHG'
today = datetime.now().strftime('%Y-%m-%d')
result = None

with ErrorBoundary("获取市场趋势分析") as eb:
    if trend_analyzer is None:
        raise RuntimeError("TrendAnalyzer 未初始化")
    result = trend_analyzer.analyze_market(index_code=INDEX_CODE, date=today)

if result:
    # 定义趋势方向映射
    def get_trend_emoji(score):
        if score > 60: return "🟢⬆️ 强势上涨"
        elif score > 30: return "🟢↗️ 上涨趋势"
        elif score > 10: return "🟡↗️ 弱势上涨"
        elif score > -10: return "🟡➡️ 震荡盘整"
        elif score > -30: return "🟠↘️ 弱势下跌"
        elif score > -60: return "🔴↘️ 下跌趋势"
        else: return "🔴⬇️ 强势下跌"
    
    # 定义仓位建议
    def get_position_suggestion(score):
        if score > 50: return "80-100% (进攻配置)"
        elif score > 20: return "60-80% (积极配置)"
        elif score > 0: return "40-60% (均衡配置)"
        elif score > -20: return "20-40% (防守配置)"
        else: return "0-20% (空仓或极低仓位)"
    
    print("=" * 70)
    print("📊 市场趋势分析结果")
    print("=" * 70)
    
    # 基本信息
    print(f"\n📅 分析日期: {result.analysis_date.strftime('%Y-%m-%d %H:%M')}")
    print(f"📈 分析标的: {result.index_code}")
    
    # 综合评估
    print(f"\n{'─' * 70}")
    print("💹 综合评估")
    print(f"{'─' * 70}")
    print(f"   综合得分: {result.composite_score:+.1f} / 100")
    print(f"   趋势方向: {get_trend_emoji(result.composite_score)}")
    print(f"   市场阶段: {result.market_phase}")
    print(f"   建议仓位: {get_position_suggestion(result.composite_score)}")
    
    # 多周期分析
    print(f"\n{'─' * 70}")
    print("🔍 多周期分析")
    print(f"{'─' * 70}")
    print(f"   短期趋势 (40天):  得分={result.short_term.score:+.1f}, {result.short_term.direction.value}")
    print(f"   中期趋势 (120天): 得分={result.medium_term.score:+.1f}, {result.medium_term.direction.value}")
    print(f"   长期趋势 (240天): 得分={result.long_term.score:+.1f}, {result.long_term.direction.value}")
    
    # 周期权重说明
    print(f"\n   📝 权重分配: 短期 20% + 中期 30% + 长期 50%")
    weighted_score = result.short_term.score * 0.2 + result.medium_term.score * 0.3 + result.long_term.score * 0.5
    print(f"   📝 加权计算: {result.short_term.score:.1f}×0.2 + {result.medium_term.score:.1f}×0.3 + {result.long_term.score:.1f}×0.5 = {weighted_score:.1f}")
    
    # 共振分析
    if hasattr(result, 'resonance') and result.resonance:
        print(f"\n{'─' * 70}")
        print("🔗 多周期共振分析")
        print(f"{'─' * 70}")
        all_bull = result.resonance.get('all_bullish', False)
        all_bear = result.resonance.get('all_bearish', False)
        if all_bull:
            print("   ✅ 全周期共振看涨！趋势强劲")
        elif all_bear:
            print("   ⚠️ 全周期共振看跌！趋势极弱")
        else:
            print("   ➡️ 周期信号不一致，观望为主")
    
    print(f"\n{'=' * 70}")
else:
    print('⚠️ 无法获取市场状态，请检查数据源连接')


2026-01-01 20:11:13,288 - core.trend_analyzer - INFO - 市场趋势分析完成: 000001.XSHG, 综合得分=30.6, 阶段=突破在即


📊 市场趋势分析结果

📅 分析日期: 2026-01-01 20:11
📈 分析标的: 000001.XSHG

──────────────────────────────────────────────────────────────────────
💹 综合评估
──────────────────────────────────────────────────────────────────────
   综合得分: +30.6 / 100
   趋势方向: 🟢↗️ 上涨趋势
   市场阶段: 突破在即
   建议仓位: 60-80% (积极配置)

──────────────────────────────────────────────────────────────────────
🔍 多周期分析
──────────────────────────────────────────────────────────────────────
   短期趋势 (40天):  得分=+38.4, 上涨趋势
   中期趋势 (120天): 得分=+31.8, 上涨趋势
   长期趋势 (240天): 得分=+26.8, 弱势上涨

   📝 权重分配: 短期 20% + 中期 30% + 长期 50%
   📝 加权计算: 38.4×0.2 + 31.8×0.3 + 26.8×0.5 = 30.6

──────────────────────────────────────────────────────────────────────
🔗 多周期共振分析
──────────────────────────────────────────────────────────────────────
   ✅ 全周期共振看涨！趋势强劲



## 4. 生成趋势得分仪表盘

使用 `MarketGauge` 创建交互式仪表盘：
- 显示市场趋势综合得分（-100 到 +100）
- 可视化当前市场状态
- 使用 Plotly 生成交互式图表

In [6]:
# 仪表盘（带错误处理）
gauge = MarketGauge()

if result:
    with ErrorBoundary("创建趋势仪表盘") as eb:
        fig = gauge.create_trend_gauge(score=result.composite_score)
        fig.show()
else:
    print('⚠️ 无法创建仪表盘，趋势分析结果不可用')

## 5. 绘制 K 线图与技术指标

使用 `ChartEngine` 绘制专业 K 线图：
- 获取最近 90 天的价格数据（开高低收、成交量）
- 绘制 K 线图并叠加技术指标（如均线）
- 使用 Plotly 生成交互式图表，支持缩放和悬停查看详情

In [7]:
# K线图（带错误处理）
engine = ChartEngine()
START_DATE = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

with ErrorBoundary("获取价格数据") as eb:
    if jq is None:
        raise RuntimeError("JQData客户端未初始化")
    df = jq.get_price(INDEX_CODE, start_date=START_DATE, end_date=today, 
                      frequency='daily', fields=['open', 'high', 'low', 'close', 'volume'])

if not eb.has_error and df is not None and len(df) > 0:
    with ErrorBoundary("绘制K线图") as eb2:
        fig = engine.plot_candlestick_with_indicators(df, title=f'{INDEX_CODE} K线图')
        fig.show()
else:
    print('⚠️ 无法绘制K线图')

2026-01-01 20:12:22,233 - core.visualization.chart_engine - INFO - ChartEngine initialized with backend: plotly
2026-01-01 20:12:23,542 - jqdata.client - INFO - 获取价格数据成功: 000001.XSHG, 2025-10-03 to 2026-01-01
